[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/mechanics/lin-elastic_gradient-check.ipynb)

# Forward-Mode vs. Reverse-Mode Gradients Through the Elastic Solver

Takes the exact same composite RVE and shear-strain setup as
[`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb) and asks a different question: can we
differentiate the solve with respect to a material parameter (here, the fibre's Young's modulus
$E_\text{fibre}$), and do forward-mode (`jax.jvp`) and reverse-mode (`jax.grad`) autodiff agree on
the answer?

**Why this needs a different CG implementation than the production solver.** The production
Lippmann-Schwinger solve (`solvers.elliptic.vector.lippmann_schwinger.solve_lippmann_schwinger`,
what `problems.mechanics.solve_mechanics` calls) uses `solvers.krylov.cg.cg_solve`: a thin wrapper
around `jax.scipy.sparse.linalg.cg`, chosen for speed (`jax.lax.while_loop` early-exits as soon as
CG converges, instead of paying for a fixed iteration budget every call). That speed comes at a
cost this project has verified directly: `jax.grad` through `cg_solve` on this project's own
FFT-based operators gives a **silently wrong gradient** -- not an error, not `NaN`, just a wrong
number, off by many orders of magnitude. The root cause (see `solvers.krylov.cg.cg_solve_diff`'s
docstring for the full finding) is that this project's Green's operator is *exactly singular* at
the zero-frequency mode by construction (the mean strain is prescribed as `eps_bar`, not solved
for), and JAX's implicit-differentiation gradient rule for `cg` doesn't handle that singularity.

`solvers.krylov.cg.cg_solve_scan` fixes this by differentiating the *literal sequence of executed
CG steps* (`jax.lax.scan`, which has an ordinary, correct transpose rule) instead of trying to
invert the operator abstractly. It's what this notebook uses, and it's the same mechanism
[`lin-elastic_inverse-calibration.ipynb`](./lin-elastic_inverse-calibration.ipynb) builds its
gradient-based material-parameter fitting on.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())

## Generate the composite RVE

Identical geometry to `lin-elastic_strain.ipynb`: a square-packed 2-fibre RVE, Vf~0.5, 5 µm fibre
radius, 0.2 µm voxel spacing, one voxel thick (plane-strain). This is fixed, non-differentiated
data -- only the fibre's Young's modulus below becomes a traced parameter.

In [ ]:
from generation.rve import make_square_composite_rve

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber=0.005, dx=0.0002, N_min=32, nz=1,
)
Nv = int(np.prod(n))
dx = tuple(Li / ni for Li, ni in zip(L, n))
phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fibre

# Same prescribed macroscopic shear strain as lin-elastic_strain.ipynb.
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
])

print("grid n :", n, "  Nv:", Nv)
print("fibre volume fraction (actual):", f"{phi_act:.4f}")

## A differentiable elastic solve

There's a trap here worth naming explicitly: `materialmodels.elastic.isotropic.
LinearElasticIsotropic.__init__` does `self.E = float(E)` -- fine for a plain solve, but it
**concretizes** a traced value, so passing a `jax.grad`-traced `E_fibre` into `LinearElasticIsotropic`
raises `ConcretizationTypeError` immediately at construction. A material parameter meant to be
differentiated can't go through the production material classes at all right now.

The fix below builds the stiffness tensor directly from `(lam, mu)` via plain `jnp` arithmetic,
bypassing `LinearElasticIsotropic` entirely -- but still uses the *real* production building blocks
for the actual PDE operator (`operators.green.GreenOperatorWillot`, `operators.projection.
Gamma0Operator`, `operators.general_functions.ddot42`), so only the material-construction step,
not the Lippmann-Schwinger scheme itself, is reimplemented. `solvers.krylov.cg.cg_solve_scan`
(imported below) is what makes the whole thing differentiable.

In [ ]:
from operators.green import GreenOperatorWillot
from operators.projection import Gamma0Operator
from operators.general_functions import ddot42
from solvers.krylov.cg import cg_solve_scan

I2 = jnp.eye(3)

def stiffness(lam, mu):
    """Isotropic 4th-order stiffness from Lame parameters (3,3,3,3)."""
    return (lam * jnp.einsum('ij,kl->ijkl', I2, I2)
            + mu * (jnp.einsum('ik,jl->ijkl', I2, I2) + jnp.einsum('il,jk->ijkl', I2, I2)))

def lame(E, nu):
    return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))

def solve_elastic_diff(E_fiber, maxiter=100, toler_lin=1e-6):
    """Differentiable Lippmann-Schwinger solve: same A(v)=Gamma0(C:v), b=-Gamma0(C:eps0)
    as solve_lippmann_schwinger, with cg_solve_scan standing in for cg_solve."""
    nu_matrix, nu_fiber = 0.35, 0.20
    E_matrix = 3.0e3
    lam_m, mu_m = lame(E_matrix, nu_matrix)
    lam_f, mu_f = lame(E_fiber, nu_fiber)

    C_m, C_f = stiffness(lam_m, mu_m), stiffness(lam_f, mu_f)
    C_field = jnp.where((phase == 1)[None, None, None, None, :], C_f[..., None], C_m[..., None])

    # Reference medium: arithmetic mean, same choice build_reference_green_operator makes.
    lam0, mu0 = 0.5 * (lam_m + lam_f), 0.5 * (mu_m + mu_f)
    green_op = GreenOperatorWillot(n, L, lam0, mu0, dx)
    gamma0 = Gamma0Operator(n, green_op)

    def A_op(v_flat):
        v = v_flat.reshape(3, 3, Nv)
        return gamma0(ddot42(C_field, v)).reshape(-1)

    eps0 = jnp.ones((3, 3, Nv)) * eps_bar[:, :, None]
    sigma0 = ddot42(C_field, eps0)
    bb = -gamma0(sigma0).reshape(-1)

    x0 = jnp.zeros_like(bb)
    delta_flat, converged = cg_solve_scan(A_op, bb, x0, toler_lin, maxiter)
    delta = delta_flat.reshape(3, 3, Nv)
    eps = eps0 + delta
    sigma = ddot42(C_field, eps)
    return eps, sigma, converged

def loss(E_fiber, maxiter=100, toler_lin=1e-6):
    """Homogenized shear stress tau_xy -- the quantity we differentiate w.r.t. E_fiber."""
    eps, sigma, converged = solve_elastic_diff(E_fiber, maxiter, toler_lin)
    sigma_bar = jnp.mean(sigma, axis=-1)
    return sigma_bar[0, 1]

E_fiber0 = 70.0e3   # MPa, matching the glass fibre in lin-elastic_strain.ipynb
print("tau_xy at E_fiber = 70e3 MPa:", float(loss(E_fiber0)), "MPa")

## Compare forward-mode and reverse-mode gradients

There's no simple closed-form gradient to check against here (unlike the homogeneous sanity checks
elsewhere in this project) -- the composite is heterogeneous, so the "proof" is forward-mode,
reverse-mode, and a central finite difference all agreeing with each other, independently computed
three different ways.

In [ ]:
# Reverse-mode: jax.grad (backpropagates through cg_solve_scan's jax.lax.scan).
loss_rev, grad_rev = jax.value_and_grad(loss)(E_fiber0)

# Forward-mode: jax.jvp (propagates a tangent through the same computation).
loss_fwd, grad_fwd = jax.jvp(loss, (E_fiber0,), (1.0,))

# Central finite difference, as a third, independent cross-check.
d_E = 1.0  # MPa
grad_fd = (loss(E_fiber0 + d_E) - loss(E_fiber0 - d_E)) / (2.0 * d_E)

print(f"loss (tau_xy) at E_fiber=70e3 MPa   : {float(loss_rev):.10f}")
print()
print(f"reverse-mode (jax.grad)             : {float(grad_rev):.10e}")
print(f"forward-mode (jax.jvp)              : {float(grad_fwd):.10e}")
print(f"central finite difference           : {float(grad_fd):.10e}")
print()
rel_rev_fd  = abs(float(grad_rev) - float(grad_fd))  / abs(float(grad_fd))
rel_fwd_rev = abs(float(grad_fwd) - float(grad_rev)) / abs(float(grad_rev))
print(f"|grad_rev - grad_fd|  / |grad_fd|   = {rel_rev_fd:.2e}")
print(f"|grad_fwd - grad_rev| / |grad_rev|  = {rel_fwd_rev:.2e}")

assert rel_rev_fd < 1e-3
assert rel_fwd_rev < 1e-6
print("\nAll three agree -- cg_solve_scan's gradient through the heterogeneous solve is trustworthy.")

## Why `cg_solve` (the production, early-exit CG) can't be differentiated safely

The claim in `solvers.krylov.cg.cg_solve`'s own docstring -- that `jax.grad` through it is
*silently* wrong on this project's operators -- was stated there, not shown. Let's check it
directly: the exact same `solve_elastic_diff` above, with `cg_solve_scan` swapped for `cg_solve`
(everything else -- the operator, the RHS, the reference medium -- byte-for-byte identical).

In [ ]:
from solvers.krylov.cg import cg_solve

def solve_elastic_cgsolve(E_fiber, maxiter=100, toler_lin=1e-6):
    nu_matrix, nu_fiber = 0.35, 0.20
    E_matrix = 3.0e3
    lam_m, mu_m = lame(E_matrix, nu_matrix)
    lam_f, mu_f = lame(E_fiber, nu_fiber)
    C_m, C_f = stiffness(lam_m, mu_m), stiffness(lam_f, mu_f)
    C_field = jnp.where((phase == 1)[None, None, None, None, :], C_f[..., None], C_m[..., None])
    lam0, mu0 = 0.5 * (lam_m + lam_f), 0.5 * (mu_m + mu_f)
    green_op = GreenOperatorWillot(n, L, lam0, mu0, dx)
    gamma0 = Gamma0Operator(n, green_op)

    def A_op(v_flat):
        v = v_flat.reshape(3, 3, Nv)
        return gamma0(ddot42(C_field, v)).reshape(-1)

    eps0 = jnp.ones((3, 3, Nv)) * eps_bar[:, :, None]
    sigma0 = ddot42(C_field, eps0)
    bb = -gamma0(sigma0).reshape(-1)
    x0 = jnp.zeros_like(bb)
    delta_flat, converged = cg_solve(A_op, bb, x0, toler_lin, maxiter)   # <-- only this line differs
    delta = delta_flat.reshape(3, 3, Nv)
    eps = eps0 + delta
    sigma = ddot42(C_field, eps)
    return eps, sigma, converged

def loss_cgsolve(E_fiber, maxiter=100, toler_lin=1e-6):
    eps, sigma, converged = solve_elastic_cgsolve(E_fiber, maxiter, toler_lin)
    return jnp.mean(sigma, axis=-1)[0, 1]

# forward value must still match -- same equation, same converged answer
loss_check = loss_cgsolve(E_fiber0)
print(f"forward loss via cg_solve : {float(loss_check):.10f}  (trusted: {float(loss_rev):.10f})")
assert abs(float(loss_check) - float(loss_rev)) / abs(float(loss_rev)) < 1e-8

grad_cgsolve_rev = jax.grad(loss_cgsolve)(E_fiber0)
_, grad_cgsolve_fwd = jax.jvp(loss_cgsolve, (E_fiber0,), (1.0,))

print()
print(f"trusted gradient (cg_solve_scan)        : {float(grad_rev):.10e}")
print(f"jax.grad through cg_solve (reverse-mode) : {float(grad_cgsolve_rev):.10e}"
      f"   ({abs(float(grad_cgsolve_rev)-float(grad_rev))/abs(float(grad_rev)):.2e} relative error)")
print(f"jax.jvp  through cg_solve (forward-mode) : {float(grad_cgsolve_fwd):.10e}"
      f"   ({abs(float(grad_cgsolve_fwd)-float(grad_rev))/abs(float(grad_rev)):.2e} relative error)")

The forward pass is identical (same physics, same converged solution) -- but `jax.grad`
(reverse-mode) through `cg_solve` comes out **wrong by roughly nine orders of magnitude**, silently:
no error, no `NaN`, just a number that looks plausible until you check it against something
trustworthy. `jax.jvp` (forward-mode) through the same function is much closer, though still not
exact -- consistent with `cg_solve_diff`'s docstring finding that it's specifically the *reverse-mode
adjoint solve* that runs into this project's structurally singular operator (the Green's operator's
zero-frequency mode), not the forward/tangent direction.

This is exactly why `problems.mechanics.solve_mechanics` and friends don't expose a "just
differentiate the production solve" path today -- `cg_solve_scan` (slower, but correct) is what a
caller has to opt into, by hand, the way this notebook does.

## Speed: what differentiability actually costs

In [ ]:
import time

# Isolate the forward-pass cost alone (no jax.grad/jax.jvp) -- both JIT-compiled and warmed up
# first so compilation time doesn't pollute the timing. Same maxiter=100, same RVE, same E_fiber0
# for both, so the only difference is cg_solve_scan's fixed-length jax.lax.scan (pays for all 100
# steps, every call -- what buys it a working reverse-mode gradient) against jax.scipy.sparse.
# linalg.cg's early-exit jax.lax.while_loop.
solve_scan_jit    = jax.jit(lambda E: solve_elastic_diff(E, maxiter=100))
solve_cgsolve_jit = jax.jit(lambda E: solve_elastic_cgsolve(E, maxiter=100))

def bench(fn, repeats=10):
    out = jax.block_until_ready(fn(E_fiber0))   # warmup + compile, not timed
    t0 = time.perf_counter()
    for _ in range(repeats):
        out = jax.block_until_ready(fn(E_fiber0))
    return (time.perf_counter() - t0) / repeats * 1000, out

t_scan, _    = bench(solve_scan_jit)
t_cgsolve, _ = bench(solve_cgsolve_jit)

print(f"cg_solve_scan (fixed maxiter=100, differentiable) : {t_scan:8.2f} ms/solve")
print(f"cg_solve      (early-exit, production default)    : {t_cgsolve:8.2f} ms/solve")
print(f"slowdown factor                                    : {t_scan / t_cgsolve:8.1f}x")

## Next steps

- Swap `E_fiber` for any other material parameter (matrix `E`/`nu`, either phase's `nu`) -- nothing
  above is specific to the fibre modulus.
- [`lin-elastic_inverse-calibration.ipynb`](./lin-elastic_inverse-calibration.ipynb) is exactly this
  mechanism put to use: `cg_solve_scan`-backed gradients driving an `optax` fit of material
  parameters against a target stress-strain response.
- The `maxiter=100` fixed budget above is generous for this RVE (CG converges well before that) --
  for a much larger/higher-contrast grid, `cg_solve_scan`'s no-early-exit cost (see the speed
  comparison above) can dominate; watch for it if you reuse this pattern at scale.